# Phelan frozen BERT model refinement

This notebook reproduces the fixed 473/118 development split, compares deeper MLP heads and supervised contrastive regularization, and explores follow-up class-specific logit offsets and loss weights. It selects using validation results only. It is a research contribution for later integration into the group's single final notebook, not a replacement for the full group submission. Run from the `group project` directory with the project data present. Python package versions are recorded in the output. The first study did not use back-translation. The second study below tests training-only Neutral augmentation. The offset grid and loss-weight grid were added after previous iterations; historical test results had already been observed.

The initial selection rule required validation macro-F1, Irrelevant F1, and Neutral F1 all to meet or exceed the re-run original MLP, with at least one strict improvement. After that rule found no candidate, a follow-up tolerance of 0.005 validation macro-F1 was introduced for candidates that strictly improved both target-class F1 scores. This tolerance was chosen after inspecting the validation results and is exploratory, not preregistered. Test data are read only after selection. The test set was examined in earlier project work, so it is not a fresh holdout.

**Environment.** The recorded run used Python 3.11, PyTorch 2.6.0 with CUDA, transformers 4.57.1, pandas 2.2.3, scikit-learn 1.6.1, numpy 2.0.2, and ftfy 6.3.1. Install a compatible PyTorch build for the machine, then install the remaining libraries if needed. The pinned public BERT weights must be cached or downloaded. Run this notebook from the repository's `group project` directory. Each run writes a new timestamped output directory.


In [ ]:
import os, json, time, random, re, unicodedata, platform, copy
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
import ftfy, sklearn, transformers
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModel

PROJECT = Path.cwd()
if not (PROJECT / 'data/processed/split_manifest.csv').exists():
    if (PROJECT / 'group project/data/processed/split_manifest.csv').exists(): PROJECT /= 'group project'
    else: raise FileNotFoundError('Run this notebook from the project root or group project directory')
from datetime import datetime, timezone
RUN_TAG = os.environ.get('PHELAN_RUN_TAG') or datetime.now(timezone.utc).strftime('refinement_%Y%m%dT%H%M%SZ')
OUT = PROJECT / 'runs/phelan' / RUN_TAG
OUT.mkdir(parents=True, exist_ok=False)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
MODEL_ID = 'google-bert/bert-base-uncased'
REVISION = '86b5e0934494bd15c9632b12f734a8a67f723594'
SURFACE = ['exclaim_count','question_count','url_count','mention_count']
NAMES = ['Negative','Positive','Irrelevant','Neutral']
print('Results:', OUT, 'device:', DEVICE, flush=True)

## 1. Fixed data split and frozen embeddings
Preprocessing and pooling match the original implementation. Standardization is fitted on training examples only. The recorded BERT revision is pinned and BERT is never fine-tuned.

In [ ]:
data = PROJECT/'data/processed'
pool = pd.read_csv(data/'train_cleaned.csv',keep_default_na=False)
manifest = pd.read_csv(data/'split_manifest.csv',keep_default_na=False)
assert pool.row_id.is_unique and manifest.row_id.is_unique and set(pool.row_id)==set(manifest.row_id)
assert manifest.groupby('group_id').partition.nunique().max()==1
pool = pool.merge(manifest[['row_id','partition']],on='row_id',validate='one_to_one')
train = pool[pool.partition=='train'].reset_index(drop=True)
val = pool[pool.partition=='validation'].reset_index(drop=True)
assert len(train)==473 and len(val)==118
def normalise(s): return re.sub(r'\s+', ' ', unicodedata.normalize('NFC',ftfy.fix_text(str(s)))).strip()
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION)
bert = AutoModel.from_pretrained(MODEL_ID,revision=REVISION).to(DEVICE).eval()
for p in bert.parameters(): p.requires_grad=False
@torch.inference_mode()
def embed(frame):
    texts=[normalise(x) for x in frame.Phrase]
    chunks=[]
    for start in range(0,len(texts),64):
        batch=tokenizer(texts[start:start+64],padding=True,truncation=True,max_length=128,return_tensors='pt').to(DEVICE)
        hidden=bert(**batch).last_hidden_state
        mask=batch['attention_mask'].unsqueeze(-1).float()
        chunks.append(((hidden*mask).sum(1)/mask.sum(1).clamp(min=1e-9)).cpu().numpy())
    return np.concatenate(chunks)
scaler=StandardScaler().fit(train[SURFACE].astype(float))
def features(frame): return np.concatenate([embed(frame),scaler.transform(frame[SURFACE].astype(float))],axis=1)
Xtr=features(train); Xval=features(val)
ytr=train.Sentiment.to_numpy(dtype=int); yval=val.Sentiment.to_numpy(dtype=int)
assert Xtr.shape==(473,772) and Xval.shape==(118,772)
print('Embeddings ready',Xtr.shape,Xval.shape,flush=True)

## 2. Model variants and supervised contrastive loss
Each example is an anchor; other same-label training examples are positives and different-label examples are negatives. The loss acts on the normalized output of the final MLP hidden layer. It is a supervised contrastive regularizer inspired by Khosla et al. (2020), jointly trained with weighted cross-entropy; the BERT encoder remains frozen. All training is full-batch and uses the same original data and seed.

In [ ]:
SPECS = [
 {'name':'original_reproduced','hidden':[512,256],'dropout':0.5,'contrastive_weight':0.0},
 {'name':'deep_512_256_128','hidden':[512,256,128],'dropout':0.5,'contrastive_weight':0.0},
 {'name':'deep_512_256_128_64','hidden':[512,256,128,64],'dropout':0.5,'contrastive_weight':0.0},
 {'name':'supcon_005','hidden':[512,256],'dropout':0.5,'contrastive_weight':0.05},
 {'name':'supcon_015','hidden':[512,256],'dropout':0.5,'contrastive_weight':0.15},
 {'name':'supcon_030','hidden':[512,256],'dropout':0.5,'contrastive_weight':0.30},
]
for i_mult in [0.75,1.0,1.25]:
    for n_mult in [0.75,1.0,1.25]:
        if i_mult==1.0 and n_mult==1.0: continue
        SPECS.append({'name':f'loss_weight_irrel_{i_mult:.2f}_neutral_{n_mult:.2f}',
                      'hidden':[512,256],'dropout':0.5,'contrastive_weight':0.0,
                      'class_weight_multipliers':[1.,1.,i_mult,n_mult]})
class Head(nn.Module):
    def __init__(self,hidden,dropout):
        super().__init__(); layers=[]; prev=772
        for h in hidden:
            layers += [nn.Linear(prev,h),nn.ReLU(),nn.Dropout(dropout)]; prev=h
        self.encoder=nn.Sequential(*layers); self.classifier=nn.Linear(prev,4)
    def forward(self,x):
        z=self.encoder(x); return self.classifier(z),z
def supervised_contrastive(z,y,temperature=0.1):
    z=F.normalize(z,dim=1); logits=z@z.T/temperature
    diag=torch.eye(len(y),dtype=torch.bool,device=y.device)
    pos=(y[:,None]==y[None,:]) & ~diag
    if (pos.sum(1)==0).any(): raise ValueError('Each anchor requires a positive')
    logits=logits-logits.max(dim=1,keepdim=True).values.detach()
    log_prob=logits-torch.logsumexp(logits.masked_fill(diag,-float('inf')),dim=1,keepdim=True)
    return -((log_prob.masked_fill(~pos,0).sum(1))/pos.sum(1)).mean()
Xt=torch.tensor(Xtr,dtype=torch.float32,device=DEVICE); yt=torch.tensor(ytr,dtype=torch.long,device=DEVICE)
Xv=torch.tensor(Xval,dtype=torch.float32,device=DEVICE)
weights=compute_class_weight('balanced',classes=np.arange(4),y=ytr)
loss_fn=nn.CrossEntropyLoss(weight=torch.tensor(weights,dtype=torch.float32,device=DEVICE))
def evaluate(y,p):
    r=classification_report(y,p,labels=[0,1,2,3],target_names=NAMES,output_dict=True,zero_division=0)
    return {'accuracy':float(accuracy_score(y,p)),'macro_f1':float(r['macro avg']['f1-score']),
            'per_class':{k:{m:float(r[k][m]) for m in ['precision','recall','f1-score','support']} for k in NAMES}}
def fit(spec):
    torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
    model=Head(spec['hidden'],spec['dropout']).to(DEVICE)
    opt=torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-4)
    effective=weights*np.asarray(spec.get('class_weight_multipliers',[1.,1.,1.,1.]))
    local_loss_fn=nn.CrossEntropyLoss(weight=torch.tensor(effective,dtype=torch.float32,device=DEVICE))
    best=-1.; best_state=None; bad=0; history=[]; best_epoch=None
    for epoch in range(1,301):
        model.train(); opt.zero_grad(); logits,z=model(Xt)
        ce=local_loss_fn(logits,yt)
        sc=supervised_contrastive(z,yt) if spec['contrastive_weight'] else torch.tensor(0.,device=DEVICE)
        loss=ce+spec['contrastive_weight']*sc
        if not torch.isfinite(loss): raise ValueError('Nonfinite training loss')
        loss.backward(); opt.step(); model.eval()
        with torch.inference_mode(): pred=model(Xv)[0].argmax(1).cpu().numpy()
        score=evaluate(yval,pred)
        history.append({'epoch':epoch,'weighted_ce':float(ce.detach().cpu()),'contrastive':float(sc.detach().cpu()),'validation_macro_f1':score['macro_f1']})
        if score['macro_f1']>best:
            best=score['macro_f1']; best_epoch=epoch; best_state=copy.deepcopy(model.state_dict()); bad=0
        else:
            bad+=1
            if bad>=15: break
    model.load_state_dict(best_state); model.eval()
    with torch.inference_mode(): pred=model(Xv)[0].argmax(1).cpu().numpy()
    return model,evaluate(yval,pred),pred,history,best_epoch

trained={}; validation=[]
for spec in SPECS:
    t=time.time(); model,score,pred,history,epoch=fit(spec)
    trained[spec['name']]=(model,score,pred)
    record={**spec,'best_epoch':epoch,'epochs_run':len(history),'elapsed_seconds':round(time.time()-t,2),
            'validation_macro_f1':score['macro_f1'],'validation_accuracy':score['accuracy'],
            'validation_irrelevant_f1':score['per_class']['Irrelevant']['f1-score'],
            'validation_neutral_f1':score['per_class']['Neutral']['f1-score']}
    validation.append(record)
    pd.DataFrame(history).to_csv(OUT/(spec['name']+'_history.csv'),index=False)
    pd.DataFrame({'row_id':val.row_id,'true':yval,'predicted':pred}).to_csv(OUT/(spec['name']+'_validation_predictions.csv'),index=False)
    print(record,flush=True)
pd.DataFrame(validation).to_csv(OUT/'training_validation_comparison.csv',index=False)

## 3. Follow-up validation-only class bias grid and final selection
The first round found no qualifying deep or contrastive model. In the second round, we added a fixed 5 by 5 grid of additive logits for Irrelevant and Neutral on the original head, excluding the zero-zero setting. These are decision offsets, not calibrated probabilities. A third round adds eight class-weight configurations: the existing balanced weights for Irrelevant and Neutral are multiplied by 0.75, 1.0 or 1.25, excluding the unchanged pair. Compare all candidates with the same class-and-macro validation rule. Both follow-up grids were designed after previous results and the project test had already been inspected in prior work; report them as exploratory and do not claim an untouched test set.

In [ ]:
base_model=trained['original_reproduced'][0]
with torch.inference_mode(): val_logits=base_model(Xv)[0].cpu().numpy()
for irrelevant_offset in [-0.6,-0.3,0.0,0.3,0.6]:
    for neutral_offset in [-0.6,-0.3,0.0,0.3,0.6]:
        if irrelevant_offset==0 and neutral_offset==0: continue
        bias=[0.0,0.0,irrelevant_offset,neutral_offset]
        pred=(val_logits+np.asarray(bias)).argmax(1)
        score=evaluate(yval,pred)
        name='bias_irrel_%+.1f_neutral_%+.1f'%(irrelevant_offset,neutral_offset)
        trained[name]=(base_model,score,pred)
        validation.append({'name':name,'hidden':[512,256],'dropout':0.5,'contrastive_weight':0.0,
          'logit_bias':bias,'best_epoch':validation[0]['best_epoch'],'epochs_run':0,'elapsed_seconds':0.0,
          'validation_macro_f1':score['macro_f1'],'validation_accuracy':score['accuracy'],
          'validation_irrelevant_f1':score['per_class']['Irrelevant']['f1-score'],
          'validation_neutral_f1':score['per_class']['Neutral']['f1-score']})
pd.DataFrame(validation).to_csv(OUT/'validation_comparison.csv',index=False)
original=validation[0]
keys=['validation_macro_f1','validation_irrelevant_f1','validation_neutral_f1']
eligible=[x for x in validation[1:] if all(x[k]>=original[k]-1e-12 for k in keys) and any(x[k]>original[k]+1e-12 for k in keys)]
relaxed=[x for x in validation[1:] if x['validation_macro_f1']>=original['validation_macro_f1']-0.005-1e-12
         and x['validation_irrelevant_f1']>original['validation_irrelevant_f1']+1e-12
         and x['validation_neutral_f1']>original['validation_neutral_f1']+1e-12]
chosen=max(eligible,key=lambda x:x['validation_macro_f1']) if eligible else (max(relaxed,key=lambda x:x['validation_macro_f1']) if relaxed else original)
selection_kind='strict' if eligible else ('posthoc_tolerance_0.005' if relaxed else 'original')
print('Strict eligible:',[x['name'] for x in eligible],'; relaxed eligible:',[x['name'] for x in relaxed],'; selected:',chosen['name'],flush=True)
selected_model=trained[chosen['name']][0]
test=pd.read_csv(data/'test_cleaned.csv',keep_default_na=False)
assert len(test)==397 and test.row_id.is_unique
Xtest=features(test); ytest=test.Sentiment.to_numpy(dtype=int)
with torch.inference_mode():
    selected_logits=selected_model(torch.tensor(Xtest,dtype=torch.float32,device=DEVICE))[0].cpu().numpy()
selected_pred=(selected_logits+np.asarray(chosen.get('logit_bias',[0.,0.,0.,0.]))).argmax(1)
selected_test=evaluate(ytest,selected_pred)
baseline=make_pipeline(TfidfVectorizer(ngram_range=(1,2),min_df=2,sublinear_tf=True),LogisticRegression(C=1,solver='lbfgs',max_iter=2000,random_state=7))
baseline.fit(train.text_clean,train.Sentiment)
base_pred=baseline.predict(test.text_clean)
base_test=evaluate(ytest,base_pred)
original_file=PROJECT/'runs/phelan/20261115T060046Z/test_predictions.csv'
original_preds=pd.read_csv(original_file,keep_default_na=False)
assert original_preds.row_id.is_unique and set(original_preds.row_id)==set(test.row_id)
aligned=test[['row_id','Sentiment']].merge(original_preds[['row_id','Sentiment','predicted_id']],on='row_id',validate='one_to_one',suffixes=('_test','_recorded'))
assert (aligned.Sentiment_test.astype(int)==aligned.Sentiment_recorded.astype(int)).all()
old_pred=aligned.predicted_id.to_numpy(dtype=int)
old_test=evaluate(ytest,old_pred)
comparison=[]
for label,b,old,new in zip(NAMES,[base_test['per_class'][x] for x in NAMES],[old_test['per_class'][x] for x in NAMES],[selected_test['per_class'][x] for x in NAMES]):
    row={'class':label,'support':int(b['support'])}
    for metric in ['precision','recall','f1-score']:
        key=metric.replace('-score',''); row['baseline_'+key]=b[metric];row['original_'+key]=old[metric];row['selected_'+key]=new[metric];row['original_minus_baseline_'+key]=old[metric]-b[metric];row['selected_minus_original_'+key]=new[metric]-old[metric]
    comparison.append(row)
pd.DataFrame(comparison).to_csv(OUT/'per_class_comparison.csv',index=False)
pd.DataFrame({'row_id':test.row_id,'true':ytest,'baseline':base_pred,'original_recorded':old_pred,'selected':selected_pred}).to_csv(OUT/'test_predictions.csv',index=False)
torch.save(selected_model.state_dict(),OUT/'selected_mlp_weights.pt')
np.savez(OUT/'surface_scaler.npz',mean=scaler.mean_,scale=scaler.scale_)
result={'selection_rule':'first require macro-F1 and both target class F1 not below original; if none, apply post-hoc 0.005 macro tolerance and require both target F1 strictly improve; maximize validation macro-F1; otherwise retain original',
 'validation_results':validation,'eligible':[x['name'] for x in eligible],'relaxed_eligible':[x['name'] for x in relaxed],'selection_kind':selection_kind,'selected':chosen,
 'test_baseline':base_test,'test_original_recorded':old_test,'test_selected':selected_test,
 'environment':{'python':platform.python_version(),'torch':torch.__version__,'transformers':transformers.__version__,'sklearn':sklearn.__version__,'numpy':np.__version__,'device':DEVICE,'gpu':torch.cuda.get_device_name(0) if DEVICE=='cuda' else None},
 'bert_revision':REVISION,'surface_features':SURFACE,'seed':SEED,'bias_grid_added_after_round_one':True,'weight_grid_added_after_round_two':True,
 'test_set_note':'Test results were observed in previous work; not an untouched holdout.'}
(OUT/'results.json').write_text(json.dumps(result,indent=2),encoding='utf-8')
print('Baseline:',base_test['macro_f1'],'original:',old_test['macro_f1'],'selected:',selected_test['macro_f1'],flush=True)
print(pd.DataFrame(comparison).to_string(index=False),flush=True)

## 4. Conditional paired bootstrap on the existing test set

This resamples test rows jointly for both models. The interval only addresses test-sample variation for the two fixed trained models. It does not account for selection, repeated validation searches, training-seed variation, or the prior use of the test set; therefore it is not a confirmation of a general improvement.


In [ ]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

if 'OUT' not in globals(): OUT=Path('runs/phelan/20261121_refinement_v4')
saved=pd.read_csv(OUT/'test_predictions.csv')
truth=saved['true'].to_numpy(dtype=int)
old=saved['original_recorded'].to_numpy(dtype=int)
new=saved['selected'].to_numpy(dtype=int)
assert len(truth)==397
def bootstrap_metrics(y,p):
    cm=np.bincount(4*y+p,minlength=16).reshape(4,4)
    tp=np.diag(cm); denom=cm.sum(0)+cm.sum(1)
    f1=np.divide(2*tp,denom,out=np.zeros(4,dtype=float),where=denom!=0)
    return np.array([f1.mean(),f1[2],f1[3],tp.sum()/cm.sum()])
rng=np.random.default_rng(42)
draws=np.empty((5000,4))
for b in range(len(draws)):
    ix=rng.integers(0,len(truth),len(truth))
    draws[b]=bootstrap_metrics(truth[ix],new[ix])-bootstrap_metrics(truth[ix],old[ix])
observed=bootstrap_metrics(truth,new)-bootstrap_metrics(truth,old)
metric_names=['macro_f1','irrelevant_f1','neutral_f1','accuracy']
bootstrap={'replicates':len(draws),'seed':42,'sampling':'paired non-stratified row bootstrap',
           'comparison':'selected minus original',
           'caveat':'Conditional on fixed models; excludes selection, training-seed and prior test-use uncertainty.',
           'metrics':{name:{'observed_delta':float(observed[i]),'ci_95_percentile':[float(x) for x in np.quantile(draws[:,i],[0.025,0.975])]} for i,name in enumerate(metric_names)}}
(OUT/'paired_bootstrap.json').write_text(json.dumps(bootstrap,indent=2),encoding='utf-8')
print(json.dumps(bootstrap,indent=2))

## 4. References and integration
Devlin et al. (2019), BERT: https://aclanthology.org/N19-1423/ . Khosla et al. (2020), Supervised Contrastive Learning: https://arxiv.org/abs/2004.11362 . The latter originally studies supervised contrastive representations in vision; this notebook adapts that principle to the MLP hidden representation and does not claim to reproduce the paper's training protocol. Integrate these cells and their data-loading/cleaning dependencies into the group's single final notebook after the other contributions are received.

# Second study · representation, transfer and Neutral augmentation

This section is independently runnable from its setup cell. The cells above preserve the earlier experiments. Use Python 3.11 with the earlier dependencies plus `vaderSentiment==3.3.2`, `sentencepiece==0.2.1` and `sacremoses==0.1.1`. The fixed public checkpoints can be downloaded on first use or pre-cached for offline use. Run inside the checkout with the shared processed files and the published v4 comparison artifacts. The encoder is frozen throughout. Only the team's four-way MLP is trained.

**Selection specified before scoring this round.** Use four stratified group folds within the 473 training tweets, with seeds 17, 42 and 73 and mean class probabilities. Every model receives 36 full-batch Adam steps, the previously published v4 stopping epoch. Compare controlled feature ablations, external SST-2 sentiment transfer, and a matched Neutral duplication/back-translation pair. Rank out-of-fold (OOF) ensembles by macro-F1, prioritizing those at least matching the control's two target-class F1s. Take the top three alternatives to official validation. A challenger qualifies for replacement only if its validation macro-F1, Irrelevant F1 and Neutral F1 all match or exceed the published v4 values. Among qualifying finalists choose highest validation macro-F1, breaking ties by OOF rank. If none qualifies, retain v4 and characterize the highest-validation finalist as exploratory. Select and persist the decision before loading any new test rows. No further search after this test evaluation.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import hashlib
import json
import os
import re
import unicodedata
import gc
import ftfy
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from transformers import AutoModel, AutoTokenizer, MarianMTModel, MarianTokenizer
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

root = next((p for p in (Path.cwd(), *Path.cwd().parents, Path.cwd() / "group project")
             if (p / "data/processed/split_manifest.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("Run inside the project or from the checkout root.")
data_dir = root / "data/processed"
v4 = root / "runs/phelan/20261121_refinement_v4"
new_tag = os.environ.get("PHELAN_STUDY_TAG") or datetime.now(timezone.utc).strftime("transfer_%Y%m%dT%H%M%S%fZ")
study = root / "runs/phelan" / new_tag
study.mkdir(parents=True, exist_ok=False)
def save_json(name, obj):
    (study / name).write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
labels = ["Negative", "Positive", "Irrelevant", "Neutral"]
seeds = [17, 42, 73]
checkpoints = {
    "base": ("google-bert/bert-base-uncased", "86b5e0934494bd15c9632b12f734a8a67f723594"),
    "sst": ("textattack/bert-base-uncased-SST-2", "95f0f6f859b35c8ff0863ae3cd4e2dbc702c0ae2"),
    "en_fr": ("Helsinki-NLP/opus-mt-en-fr", "dd7f6540a7a48a7f4db59e5c0b9c42c8eea67f18"),
    "fr_en": ("Helsinki-NLP/opus-mt-fr-en", "c4aed37b318c763fd177aa449b44e3b783cc6c02"),
}
# Each tuple specifies the BERT representation, auxiliary features and augmentation.
specs = {
    "control": ("base", "surface", "none"),
    "no_surface": ("base", "none", "none"),
    "style": ("base", "style", "none"),
    "lexicon": ("base", "lexicon", "none"),
    "style_lexicon": ("base", "all", "none"),
    "last_four": ("last4", "all", "none"),
    "sst_surface": ("sst", "surface", "none"),
    "sst_all": ("sst", "all", "none"),
    "dual": ("dual", "all", "none"),
    "neutral_repeat": ("dual", "all", "repeat"),
    "neutral_translate": ("dual", "all", "translate"),
}
incumbent_row = pd.read_csv(v4 / "training_validation_comparison.csv").set_index("name").loc["loss_weight_irrel_1.25_neutral_0.75"]
gate = {k: float(incumbent_row["validation_" + k]) for k in ["macro_f1", "irrelevant_f1", "neutral_f1"]}
plan = dict(models=specs, seeds=seeds, folds=4, fold_seed=1205, epochs=36, gate=gate,
            optimizer="full batch Adam lr=0.001 decay=0.0001", weight_multipliers=[1,1,1.25,0.75],
            shortlist="top 3 alternatives: both OOF target F1s >= control first, then largest OOF macro-F1; ties use declaration order",
            selection="all 3 official-validation metrics >= v4; rank by validation macro-F1 then shortlist order; if none passes, characterize top finalist and retain v4",
            checkpoints=checkpoints)
save_json("protocol.json", plan)
save_json("environment.json", {"device":str(device), "gpu":torch.cuda.get_device_name(0) if device.type=="cuda" else None,
                               "packages":{p:version(p) for p in ["torch","transformers","scikit-learn","numpy","pandas","vaderSentiment","ftfy","sentencepiece","sacremoses"]}})
manifest = pd.read_csv(data_dir/"split_manifest.csv")
pool = pd.read_csv(data_dir/"train_cleaned.csv", keep_default_na=False)
assert manifest.row_id.is_unique and pool.row_id.is_unique and set(pool.row_id)==set(manifest.row_id)
assert manifest.groupby("group_id").partition.nunique().eq(1).all()
pool = pool.merge(manifest, on=["row_id","Sentiment"], validate="one_to_one")
assert len(pool)==591
learn = pool.loc[pool.partition.eq("train")].reset_index(drop=True)
valid = pool.loc[pool.partition.eq("validation")].reset_index(drop=True)
assert (len(learn),len(valid))==(473,118)
target = learn.Sentiment.to_numpy()
folds = list(StratifiedGroupKFold(4,shuffle=True,random_state=1205).split(learn, target, learn.group_id))
assignment = np.full(len(learn), -1)
for f,(fit_ids,held) in enumerate(folds):
    assert set(learn.group_id.iloc[fit_ids]).isdisjoint(learn.group_id.iloc[held])
    assignment[held]=f
assert np.all(assignment>=0)
learn[["row_id","group_id","Sentiment"]].assign(fold=assignment).to_csv(study/"folds.csv", index=False)
print(study.name, device, "Training fold sizes:", np.bincount(assignment).tolist())

20261121_transfer_study cuda Training fold sizes: [119, 118, 118, 118]


### Neutral augmentation and its control

Only original Neutral rows in the training partition enter the local English–French–English translation pipeline. The filter rejects unchanged strings, extreme length changes, low word overlap, altered negation or protected spans, and large VADER compound-score shifts. The paired control duplicates precisely the same eligible parents. Each fold sees only translations of its own fit rows. The label is inherited, so semantic drift remains possible. The audit file retains exclusions and their reasons.

In [2]:
def clean_string(text):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", ftfy.fix_text(str(text)))).strip()

@torch.inference_mode()
def translate_batch(texts, key):
    model_id, revision = checkpoints[key]
    tok = MarianTokenizer.from_pretrained(model_id, revision=revision)
    mt = MarianMTModel.from_pretrained(model_id, revision=revision).to(device).eval()
    result = []
    for start in range(0, len(texts), 8):
        batch = tok(texts[start:start+8], padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
        output = mt.generate(**batch, do_sample=False, num_beams=4, max_new_tokens=128)
        result.extend(tok.batch_decode(output, skip_special_tokens=True))
    return result

vader = SentimentIntensityAnalyzer()
url_re = re.compile(r"(?ix)(?:https?://|www\.)\S+|\b[\w.-]+\.(?:com|org|net|io|co|uk|it|be|ly|me|tv|gg|tt)(?:/\S*)?|\b[\w.-]+\.(?:org|co)\.in(?:/\S*)?")
handle_re = re.compile(r"@\s*([A-Za-z0-9_]{2,})")
def protected(text):
    lower = text.lower()
    return [sorted(pattern.findall(lower)) for pattern in
            [url_re, handle_re, re.compile(r"#\s*(\w+)"), re.compile(r"\d+")]]

def filter_pair(src, dst):
    a,b = src.lower(),dst.lower()
    words = lambda s: set(re.findall(r"\w+",s))
    wa,wb = words(a),words(b)
    if not wb or re.sub(r"\W+", "", a)==re.sub(r"\W+", "", b):
        return "empty_or_unchanged"
    if protected(src)!=protected(dst):
        return "protected_span_changed"
    neg = re.compile(r"\b(?:no|not|never|nothing|neither|without|nobody|nowhere)\b|n['’]t\b")
    if len(neg.findall(a))!=len(neg.findall(b)):
        return "negation_changed"
    ratio = len(b.split())/max(1,len(a.split()))
    if not 0.65<=ratio<=1.5 or len(wa&wb)/max(1,len(wa|wb))<0.40:
        return "length_or_overlap"
    if abs(vader.polarity_scores(src)["compound"]-vader.polarity_scores(dst)["compound"])>0.35:
        return "polarity_shift"
    return "pass"

neutral = learn.loc[learn.Sentiment.eq(3),["row_id","Phrase"]].copy()
neutral["source"] = neutral.Phrase.map(clean_string)
neutral["french"] = translate_batch(neutral.source.tolist(), "en_fr")
neutral["english"] = [clean_string(t) for t in translate_batch(neutral.french.tolist(), "fr_en")]
neutral["filter"] = [filter_pair(a,b) for a,b in zip(neutral.source,neutral.english)]
neutral.to_csv(study/"translation_audit.csv",index=False)
accepted = neutral[neutral["filter"].eq("pass")]
assert not accepted.empty
print("Neutral round trips:",len(neutral),"Filter outcomes:",neutral["filter"].value_counts().to_dict())

Neutral round trips: 121 Filter outcomes: {'pass': 76, 'empty_or_unchanged': 13, 'protected_span_changed': 12, 'polarity_shift': 11, 'length_or_overlap': 5, 'negation_changed': 4}


### Feature ablations and encoders

The raw base checkpoint and the external SST-2 BERT provide separate 768-dimensional mean-pooled embeddings. The SST-2 binary classifier is discarded. The dual representation concatenates both vectors and trains a new four-class MLP. The layer ablation averages the last four hidden layers of base BERT. Auxiliary features include the four historical counts, five style measures and four VADER values. All scalers are fitted on original fit rows within each fold.

In [3]:
base_cols = ["exclaim_count","question_count","url_count","mention_count"]
style_cols = ["log_words","log_chars","caps_fraction","hashtags","elongation"]
lex_cols = ["neg","neu","pos","compound"]
def aux_features(frame):
    rows = []
    for rec in frame.to_dict("records"):
        text = clean_string(rec["Phrase"])
        letters = [c for c in text if c.isalpha()]
        counts = [text.count("!"),text.count("?"),len(url_re.findall(text)),len(handle_re.findall(text))]
        row = dict(zip(base_cols, [rec.get(col,x) for col,x in zip(base_cols,counts)]))
        row.update(dict(log_words=np.log1p(len(text.split())),log_chars=np.log1p(len(text)),
                        caps_fraction=sum(c.isupper() for c in letters)/max(len(letters),1),
                        hashtags=len(re.findall(r"#\s*\w+",text)),elongation=int(bool(re.search(r"([a-zA-Z])\1{2,}",text)))))
        row.update(vader.polarity_scores(text))
        rows.append(row)
    return pd.DataFrame(rows)
@torch.inference_mode()
def make_features(frame, representations):
    texts = frame.Phrase.map(clean_string).tolist()
    out = {"aux":aux_features(frame)}
    for key, names in [("base", {"base","last4","dual"}),("sst",{"sst","dual"})]:
        if not set(representations)&names:
            continue
        model_id, revision = checkpoints[key]
        tok = AutoTokenizer.from_pretrained(model_id, revision=revision)
        encoder = AutoModel.from_pretrained(model_id, revision=revision).to(device).eval()
        parts = []
        late = []
        for start in range(0,len(texts),32):
            batch = tok(texts[start:start+32], padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)
            want_layers = key=="base" and "last4" in representations
            result = encoder(**batch,output_hidden_states=want_layers)
            mask = batch.attention_mask.unsqueeze(-1).float()
            def average_tokens(states):
                return ((states*mask).sum(1)/mask.sum(1)).cpu().numpy()
            parts.append(average_tokens(result.last_hidden_state))
            if want_layers:
                late.append(average_tokens(torch.stack(result.hidden_states[-4:]).mean(0)))
        out[key] = np.concatenate(parts)
        if late:
            out["last4"] = np.concatenate(late)
        del encoder
    if "dual" in representations:
        out["dual"] = np.hstack([out["base"],out["sst"]])
    return out

aug_frame = accepted[["row_id","english"]].rename(columns={"english":"Phrase"}).reset_index(drop=True)
all_reps = {items[0] for items in specs.values()}
train_feat = make_features(learn, all_reps)
aug_feat = make_features(aug_frame, {"dual"})
features_by_set = {
    "none":[], "surface":base_cols, "style":base_cols+style_cols,
    "lexicon":base_cols+lex_cols, "all":base_cols+style_cols+lex_cols,
}
def matrix(feat, indices, rep, cols, scaler):
    embed = feat[rep][indices]
    if not cols:
        return embed.astype("float32")
    standardized = scaler.transform(feat["aux"].iloc[indices][cols])
    return np.hstack([embed,standardized]).astype("float32")
def training_arrays(name, ids):
    rep,feature_set,augmentation = specs[name]
    cols = features_by_set[feature_set]
    scaler = StandardScaler().fit(train_feat["aux"].iloc[ids][cols]) if cols else None
    X = matrix(train_feat, ids, rep, cols, scaler)
    y = target[ids]
    extras = []
    if augmentation!="none":
        fit_parents = dict(zip(learn.row_id.iloc[ids],ids))
        # Match the same eligible parent rows in the repetition and translation arms.
        seen = {clean_string(s).casefold() for s in learn.Phrase.iloc[ids]}
        parent_ids = []
        for i, item in enumerate(aug_frame.itertuples()):
            key = item.Phrase.casefold()
            if item.row_id in fit_parents and key not in seen:
                seen.add(key)
                extras.append(i)
                parent_ids.append(fit_parents[item.row_id])
        from_parents = augmentation=="repeat"
        extra_x = matrix(train_feat if from_parents else aug_feat, parent_ids if from_parents else extras, rep, cols, scaler)
        X = np.vstack([X,extra_x])
        y = np.r_[y,np.full(len(extras),3,dtype=int)]
    # Original fit-label frequencies keep loss weights matched across augmentation arms.
    weight = len(ids)/(4*np.bincount(target[ids],minlength=4))*np.array([1.,1.,1.25,0.75])
    return X,y,weight,scaler,len(extras)

def build_head(width):
    return nn.Sequential(nn.Linear(width,512),nn.ReLU(),nn.Dropout(0.5),
                         nn.Linear(512,256),nn.ReLU(),nn.Dropout(0.5),nn.Linear(256,4))
def train_heads(name,ids):
    X,y,weight,scaler,n_extra = training_arrays(name,ids)
    X = torch.from_numpy(X).to(device)
    y = torch.as_tensor(y,dtype=torch.long,device=device)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(weight,dtype=torch.float32,device=device))
    heads = []
    for seed in seeds:
        torch.manual_seed(seed)
        model = build_head(X.shape[1]).to(device)
        optimizer = torch.optim.Adam(model.parameters(),lr=0.001,weight_decay=0.0001)
        for _ in range(36):
            optimizer.zero_grad()
            loss = loss_fn(model(X),y)
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite loss")
            loss.backward()
            optimizer.step()
        heads.append(model.eval())
    return heads,scaler,n_extra

@torch.inference_mode()
def probabilities(name,heads,scaler,feat,ids):
    rep,feature_set,_ = specs[name]
    X = torch.from_numpy(matrix(feat,ids,rep,features_by_set[feature_set],scaler)).to(device)
    return np.stack([m(X).softmax(1).cpu().numpy() for m in heads])
def scores(y,pred):
    cm = confusion_matrix(y,pred,labels=np.arange(4))
    tp = np.diag(cm)
    den = cm.sum(0)+cm.sum(1)
    f1 = np.divide(2*tp,den,out=np.zeros(4),where=den>0)
    return {"macro_f1":float(f1.mean()),"accuracy":float(tp.sum()/len(y)),
            **{n.lower()+"_f1":float(v) for n,v in zip(labels,f1)}}

In [4]:
oof_rows, fold_rows, oof_table = [],[],[]
for name in specs:
    pred_probs = np.full((len(seeds),len(learn),4),np.nan,dtype="float32")
    for f,(ids,held) in enumerate(folds):
        heads,scaler,n_extra = train_heads(name,ids)
        prob = probabilities(name,heads,scaler,train_feat,held)
        pred_probs[:,held,:] = prob
        for seed,single in zip(seeds,prob):
            fold_rows.append(dict(model=name,fold=f,seed=seed,added=n_extra,**scores(target[held],single.argmax(1))))
    assert np.isfinite(pred_probs).all()
    mean_prob = pred_probs.mean(0)
    oof = scores(target,mean_prob.argmax(1))
    single_scores = [scores(target,pr.argmax(1))["macro_f1"] for pr in pred_probs]
    oof_rows.append(dict(model=name,**oof,seed_macro_mean=np.mean(single_scores),seed_macro_sd=np.std(single_scores,ddof=1)))
    oof_table.append(pd.DataFrame(dict(model=name,row_id=learn.row_id,fold=assignment,true=target,
                                       prediction=mean_prob.argmax(1),**{f"seed_{s}":p.argmax(1) for s,p in zip(seeds,pred_probs)})))
    print(name,{k:round(oof[k],4) for k in gate},flush=True)
oof_results = pd.DataFrame(oof_rows).set_index("model")
oof_results.to_csv(study/"oof_summary.csv")
pd.concat(oof_table).to_csv(study/"oof_predictions.csv",index=False)
pd.DataFrame(fold_rows).to_csv(study/"fold_seed_metrics.csv",index=False)
alternative = oof_results.drop("control").copy()
alternative["target_floor"] = alternative[["irrelevant_f1","neutral_f1"]].ge(oof_results.loc["control",["irrelevant_f1","neutral_f1"]]).all(axis=1)
alternative["order"] = np.arange(len(alternative))
shortlist = alternative.sort_values(["target_floor","macro_f1","order"],ascending=[False,False,True]).head(3).index.tolist()
save_json("shortlist.json",shortlist)
print("OOF finalists:",shortlist)

control {'macro_f1': 0.4494, 'irrelevant_f1': 0.3114, 'neutral_f1': 0.3927}
no_surface {'macro_f1': 0.4672, 'irrelevant_f1': 0.323, 'neutral_f1': 0.4292}
style {'macro_f1': 0.4542, 'irrelevant_f1': 0.3125, 'neutral_f1': 0.4229}
lexicon {'macro_f1': 0.4598, 'irrelevant_f1': 0.2893, 'neutral_f1': 0.4215}
style_lexicon {'macro_f1': 0.459, 'irrelevant_f1': 0.2981, 'neutral_f1': 0.4089}
last_four {'macro_f1': 0.4599, 'irrelevant_f1': 0.3038, 'neutral_f1': 0.4229}
sst_surface {'macro_f1': 0.4589, 'irrelevant_f1': 0.2895, 'neutral_f1': 0.4095}
sst_all {'macro_f1': 0.4432, 'irrelevant_f1': 0.2207, 'neutral_f1': 0.4344}
dual {'macro_f1': 0.4557, 'irrelevant_f1': 0.2794, 'neutral_f1': 0.4242}
neutral_repeat {'macro_f1': 0.4599, 'irrelevant_f1': 0.2707, 'neutral_f1': 0.4173}
neutral_translate {'macro_f1': 0.4539, 'irrelevant_f1': 0.2748, 'neutral_f1': 0.4198}
OOF finalists: ['no_surface', 'style', 'neutral_repeat']


### Fixed validation and test decision

Fit only the three finalists on all 473 training parents. Each averages the predictions of three independently initialized MLP heads. The validation table includes the outcome of the rule fixed at the beginning of this study. The new test evaluation loads only the selected ensemble. Historical predictions are joined by row ID to quantify paired changes and compare the original and tuned baselines.

In [5]:
valid_feat = make_features(valid, {specs[n][0] for n in shortlist})
valid_rows, valid_preds, finalist_heads = [],[],{}
for name in shortlist:
    heads,scaler,n_extra = train_heads(name,np.arange(len(learn)))
    prob = probabilities(name,heads,scaler,valid_feat,np.arange(len(valid)))
    pred = prob.mean(0).argmax(1)
    result = scores(valid.Sentiment,pred)
    passes = all(result[k]>=gate[k]-1e-12 for k in gate)
    valid_rows.append(dict(model=name,qualifies=passes,added=n_extra,**result))
    valid_preds.append(pd.DataFrame(dict(model=name,row_id=valid.row_id,true=valid.Sentiment,prediction=pred,
                                      **{f"seed_{s}":p.argmax(1) for s,p in zip(seeds,prob)})))
    finalist_heads[name]=(heads,scaler)
validation_results = pd.DataFrame(valid_rows).set_index("model")
validation_results.to_csv(study/"validation.csv")
pd.concat(valid_preds).to_csv(study/"validation_predictions.csv",index=False)
winner = validation_results.sort_values(["qualifies","macro_f1"],ascending=False,kind="stable").index[0]
promote = bool(validation_results.loc[winner,"qualifies"])
save_json("selection.json",dict(shortlist=shortlist,challenger=winner,validation_gate_pass=promote,
                                primary=winner if promote else "published_v4",validation_gate=gate))
print(validation_results.round(4).to_string())
print("Locked challenger:",winner,"Replacement criteria met:",promote)
heads, scaler = finalist_heads[winner]
torch.save([m.cpu().state_dict() for m in heads],study/"challenger_heads.pt")
if scaler is not None:
    np.savez(study/"challenger_scaler.npz",columns=np.asarray(scaler.feature_names_in_,dtype=str),mean=scaler.mean_,scale=scaler.scale_)

                qualifies  added  macro_f1  accuracy  negative_f1  positive_f1  irrelevant_f1  neutral_f1
model                                                                                                    
no_surface          False      0    0.4679    0.4831       0.5867       0.5000         0.3200      0.4651
style               False      0    0.4495    0.4661       0.5753       0.5000         0.3137      0.4091
neutral_repeat      False     76    0.4635    0.4831       0.5753       0.4615         0.3000      0.5172
Locked challenger: no_surface Replacement criteria met: False


In [6]:
test_frame = pd.read_csv(data_dir/"test_cleaned.csv",keep_default_na=False)
assert len(test_frame)==397 and test_frame.row_id.is_unique
rep, aux_name,_ = specs[winner]
test_feat = make_features(test_frame,{rep})
x_test = matrix(test_feat,np.arange(len(test_frame)),rep,features_by_set[aux_name],scaler)
states = torch.load(study/"challenger_heads.pt",map_location=device,weights_only=True)
loaded = [build_head(x_test.shape[1]).to(device) for _ in states]
for net,state in zip(loaded,states):
    net.load_state_dict(state)
    net.eval()
# Reload scaler parameters as well. Check equivalence to the in-memory model.
saved_cols = features_by_set[aux_name]
with torch.inference_mode():
    if saved_cols:
        with np.load(study/"challenger_scaler.npz",allow_pickle=False) as saved_scaler:
            assert list(saved_scaler["columns"])==saved_cols
            transformed = (test_feat["aux"][saved_cols].to_numpy()-saved_scaler["mean"])/saved_scaler["scale"]
            restored_x = np.hstack([test_feat[rep],transformed]).astype("float32")
        np.testing.assert_allclose(x_test,restored_x,rtol=1e-6,atol=1e-7)
    else:
        restored_x = x_test
    tensor = torch.from_numpy(restored_x).to(device)
    probs = torch.stack([m(tensor).softmax(1) for m in loaded]).mean(0).cpu().numpy()
    before = probabilities(winner,[m.to(device) for m in heads],scaler,test_feat,np.arange(len(test_frame))).mean(0)
    np.testing.assert_allclose(probs,before,rtol=1e-5,atol=1e-6)
pred = probs.argmax(1)
previous = pd.read_csv(v4/"test_predictions.csv").rename(columns={"selected":"v4","original_recorded":"original"})
tuned = pd.read_csv(root/"runs/baseline_refinement/test_predictions.csv").rename(columns={"Sentiment":"true","predicted":"tuned_baseline"})
paired = test_frame[["row_id","Sentiment"]].rename(columns={"Sentiment":"true"}).assign(challenger=pred)
for ref in (previous,tuned):
    paired = paired.merge(ref,on=["row_id","true"],how="left",validate="one_to_one")
assert len(paired)==397 and not paired.isna().any().any()
paired.to_csv(study/"test_predictions.csv",index=False)
score_rows, class_rows, cms = [],[],{}
for name in ["baseline","tuned_baseline","original","v4","challenger"]:
    truth,guess = paired.true.to_numpy(),paired[name].to_numpy()
    score_rows.append(dict(model=name,**scores(truth,guess)))
    report = classification_report(truth,guess,labels=np.arange(4),target_names=labels,output_dict=True,zero_division=0)
    for label in labels:
        class_rows.append({"model":name,"class":label,**report[label]})
    cms[name] = confusion_matrix(truth,guess,labels=np.arange(4)).tolist()
pd.DataFrame(score_rows).to_csv(study/"test_summary.csv",index=False)
pd.DataFrame(class_rows).to_csv(study/"test_by_class.csv",index=False)
save_json("confusion_matrices.json",cms)
def paired_statistics(cm):
    diag = np.diag(cm)
    denom = cm.sum(1)+cm.sum(0)
    f = np.divide(2*diag,denom,out=np.zeros(4),where=denom>0)
    return np.array([f.mean(),f[2],f[3],diag.sum()/cm.sum()])
bootstrap_cols = ["macro_f1","irrelevant_f1","neutral_f1","accuracy"]
rng = np.random.default_rng(42)
truth = paired.true.to_numpy()
draw_indices = rng.integers(len(truth),size=(5000,len(truth)))
ci_rows = []
for comparator in ["baseline","v4"]:
    delta = []
    for ix in draw_indices:
        a = np.bincount(4*truth[ix]+pred[ix],minlength=16).reshape(4,4)
        b = np.bincount(4*truth[ix]+paired[comparator].to_numpy()[ix],minlength=16).reshape(4,4)
        delta.append(paired_statistics(a)-paired_statistics(b))
    point = paired_statistics(np.array(cms["challenger"]))-paired_statistics(np.array(cms[comparator]))
    for j,key in enumerate(bootstrap_cols):
        lo,hi = np.quantile(np.asarray(delta)[:,j],[0.025,0.975])
        ci_rows.append(dict(comparator=comparator,metric=key,delta=point[j],lower=lo,upper=hi))
pd.DataFrame(ci_rows).to_csv(study/"paired_bootstrap.csv",index=False)
# Inspect exact generated-text collisions after the selection decision has been saved.
def text_keys(frame):
    return {clean_string(s).casefold() for s in frame.Phrase}
seen = text_keys(learn)
kept_rows = []
for parent,phrase in aug_frame[["row_id","Phrase"]].itertuples(index=False,name=None):
    key = clean_string(phrase).casefold()
    if key not in seen:
        kept_rows.append((parent,key))
        seen.add(key)
valid_texts,test_texts = text_keys(valid),text_keys(test_frame)
save_json("augmentation_overlap.json", {
    "retained_after_train_dedup":len(kept_rows),
    "exact_validation_matches":[row for row,key in kept_rows if key in valid_texts],
    "exact_test_matches":[row for row,key in kept_rows if key in test_texts],
    "check":"normalized, case-insensitive exact text; performed only after challenger lock"
})
print(pd.DataFrame(score_rows).round(4).to_string(index=False))
print("All 397 ensemble predictions reproduced after weight and scaler reload.")

         model  macro_f1  accuracy  negative_f1  positive_f1  irrelevant_f1  neutral_f1
      baseline    0.4461    0.4912       0.5887       0.4279         0.2564      0.5114
tuned_baseline    0.4368    0.4635       0.5755       0.3791         0.3178      0.4747
      original    0.4843    0.5038       0.5920       0.5178         0.3077      0.5196
            v4    0.4971    0.5113       0.5917       0.5321         0.3399      0.5246
    challenger    0.4686    0.4786       0.5439       0.5116         0.3273      0.4914
All 397 ensemble predictions reproduced after weight and scaler reload.


### Class-level figure for the report

The plot compares the shared baseline, the previous class-weight model and the locked challenger on the same test examples. Review the precision and recall table alongside it when deciding how the classifier should route tweets for human review.

In [7]:
import math
from matplotlib import pyplot as plt
metrics = pd.DataFrame(class_rows).set_index(["model","class"])
fig,ax = plt.subplots(figsize=(7.25,2.95))
series = [("baseline","TF-IDF baseline","#64748b","o"),
          ("v4","Class-weight BERT (v4)","#1b6097","s"),
          ("challenger","Locked challenger","#a94724","D")]
for i,(key,tag,color,marker) in enumerate(series):
    ax.scatter([metrics.loc[(key,label),"f1-score"] for label in labels],
               np.arange(4)+(i-1)*.18,label=tag,c=color,marker=marker,s=45,zorder=3)
ax.set_yticks(range(4),labels)
ax.set_ylim(3.54,-.53)
ax.set_xlim(0,max(.75,math.ceil(10*metrics["f1-score"].max())/10+.03))
ax.set_xlabel("F1 on the 397 shared test messages",fontsize=10)
ax.grid(axis="x",color="#e1e6ed")
ax.set_axisbelow(True)
ax.spines[["top","right","left"]].set_visible(False)
ax.tick_params(axis="y",length=0)
ax.legend(loc="lower center",bbox_to_anchor=(.5,1),ncol=3,frameon=False,fontsize=8.8)
fig.tight_layout()
fig.savefig(study/"transfer_class_f1.png",dpi=220,bbox_inches="tight",facecolor="white")
plt.show()

### Why did deleting the four counts help?

This diagnostic was added after the training-only ablation showed a gain from removing the four counts. It does not enter shortlisting or model selection. Refit the three-seed control within each original training fold. Permute one count across its held-out rows, retain all other inputs and weights, and compare the pooled out-of-fold score to the unpermuted control. Thirty fixed permutations measure sensitivity to shuffling. Their dispersion is descriptive and does not quantify generalization uncertainty.

In [8]:
n_permutations = 30
perm_predictions = np.zeros((len(base_cols),n_permutations,len(learn)),dtype=int)
unpermuted = np.full(len(learn),-1)
for fold_no,(ids,held) in enumerate(folds):
    ctrl_heads,ctrl_scale,_ = train_heads("control",ids)
    X_held = matrix(train_feat,held,"base",base_cols,ctrl_scale)
    with torch.inference_mode():
        def control_predict(array):
            tensor = torch.from_numpy(array).to(device)
            probability = torch.stack([m(tensor).softmax(-1) for m in ctrl_heads]).mean(0)
            return probability.argmax(-1).cpu().numpy()
        unpermuted[held] = control_predict(X_held)
        for j,feature in enumerate(base_cols):
            draws = []
            rng = np.random.default_rng(5200+fold_no*10+j)
            for _ in range(n_permutations):
                changed = X_held.copy()
                changed[:,768+j] = rng.permutation(changed[:,768+j])
                draws.append(changed)
            perm_predictions[j][:,held] = control_predict(np.concatenate(draws)).reshape(n_permutations,len(held))
control_score = scores(target,unpermuted)
assert abs(control_score["macro_f1"]-oof_results.loc["control","macro_f1"])<1e-10
perm_rows = []
for j,feature in enumerate(base_cols):
    for draw,prediction in enumerate(perm_predictions[j]):
        shuffled = scores(target,prediction)
        perm_rows.append(dict(feature=feature,draw=draw,**{m:control_score[m]-shuffled[m] for m in gate}))
permutations = pd.DataFrame(perm_rows)
permutations.to_csv(study/"surface_permutations.csv",index=False)
print("Mean OOF drop when each count is shuffled (control minus shuffled):")
print(permutations.groupby("feature")[list(gate)].mean().round(4).to_string())

Mean OOF drop when each count is shuffled (control minus shuffled):
                macro_f1  irrelevant_f1  neutral_f1
feature                                            
exclaim_count    -0.0047         0.0176     -0.0121
mention_count     0.0033        -0.0008      0.0189
question_count    0.0027         0.0064     -0.0028
url_count        -0.0014        -0.0135      0.0102
